In [1]:
import sys
print(sys.version)
print(sys.executable)

3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]
C:\Users\dell\anaconda3\python.exe


In [2]:
%pip install tensorflow matplotlib numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install torch torchvision

   ---------------------------------------- 0.0/126.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/126.7 MB ? eta -:--:--
   ---------------------------------------- 1.0/126.7 MB 3.0 MB/s eta 0:00:42
    --------------------------------------- 2.1/126.7 MB 3.6 MB/s eta 0:00:35
    --------------------------------------- 3.1/126.7 MB 4.0 MB/s eta 0:00:32
   - -------------------------------------- 3.7/126.7 MB 3.9 MB/s eta 0:00:32
   - -------------------------------------- 3.9/126.7 MB 3.8 MB/s eta 0:00:33
   - -------------------------------------- 4.5/126.7 MB 3.2 MB/s eta 0:00:39
   - -------------------------------------- 5.2/126.7 MB 3.1 MB/s eta 0:00:40
   - -------------------------------------- 6.3/126.7 MB 3.3 MB/s eta 0:00:37
   -- ------------------------------------- 7.3/126.7 MB 3.5 MB/s eta 0:00:35
   -- ------------------------------------- 7.9/126.7 MB 3.4 MB/s eta 0:00:36
   -- ------------------------------------- 7.9/126.7 MB 3.4 MB/s eta 0:00:36


In [4]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)
print("Installation successful!")

TensorFlow version: 2.22.0-rc0
Installation successful!


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets,transforms
from torch.utils.data import DataLoader

In [6]:
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,))
])

In [7]:
train_dataset=datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)
test_dataset=datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)
train_loader=DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)
test_loader=DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

100%|██████████| 9.91M/9.91M [01:28<00:00, 112kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 94.9kB/s]
100%|██████████| 1.65M/1.65M [00:07<00:00, 223kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 1.39MB/s]


In [13]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layer=nn.Sequential(
            nn.Conv2d(1,32,kernel_size=3,padding=1,),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

           
        )

        self.fc=nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7,128),
            nn.ReLU(),
            nn.Linear(128,10)
        )

    def forward(self,x):
        x=self.conv_layer(x)
        x=self.fc(x)
        return x

In [9]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = CNN().to(device)

In [10]:
model=CNN()
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [14]:
epochs=5
for epoch in range(epochs):
    model.train()
    running_loss=0.0
    correct=0.0
    total=0.0

    for image,label in train_loader:
        optimizer.zero_grad()
        output=model(image)
        loss=criterion(output,label)
        loss.backward()
        optimizer.step()
        running_loss+=loss.item()
        _, predicted=torch.max(output,1)
        total+=label.size(0)
        correct+=(predicted==label).sum().item()

    print(
        f"Epoch {epoch+1}/{epochs}, "
        f"Loss: {running_loss/len(train_loader):.4f}, "
        f"Accuracy: {100*correct/total:.2f}%"
    )

Epoch 1/5, Loss: 0.0133, Accuracy: 99.56%
Epoch 2/5, Loss: 0.0114, Accuracy: 99.62%
Epoch 3/5, Loss: 0.0094, Accuracy: 99.67%
Epoch 4/5, Loss: 0.0076, Accuracy: 99.73%
Epoch 5/5, Loss: 0.0075, Accuracy: 99.75%


In [12]:
model.eval()
total=0
correct=0
for image,label in test_loader:
    output=model(image)
    _, predicted=torch.max(output,1)
    total+=label.size(0)
    correct+=(predicted==label).sum().item()
print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 99.08%
